# Live AI Model Benchmark Notebook

This standalone notebook compares OpenAI, Anthropic, and Google models through live API calls only. It loads the same `.env` variable names as the previous tool, runs a small benchmark-aligned prompt suite, measures latency and success/failure behavior, applies a simple automated quality rubric, and displays the decision matrix directly in notebook cells.

## Live-only behavior

Running the benchmark cells sends prompts to live provider APIs and may incur cost. Start with `AIMBT_LIVE_PROMPT_CASE_LIMIT=3` if you want a smaller test run, or use `all` to run every built-in prompt.

## Evaluation Methodology

This notebook implements a lightweight text-chat evaluation workflow: each candidate model receives the same shared prompt suite, and the notebook records provider status, latency, raw output, response preview, token-usage metadata when available, and an automated quality observation. The quality score is a simple rubric-term coverage check against expected concepts for each prompt; it is useful for fast comparison and classroom analysis, but it is not a formal proof of correctness.

Interpret the automated quality score as one decision signal alongside public benchmark evidence, provider reliability, latency, and responsible-AI risk review. For high-stakes automated decisions, this notebook should be extended with stronger domain-specific checks and human review.

## Evaluation Framework Context

Industry evaluation tooling is moving quickly. RAGAS is commonly used for retrieval-augmented generation evaluation, MLflow is useful for experiment tracking and scalable evaluation pipelines, and DSPy is useful for prompt/program optimization. This notebook includes all three in a scoped way: MLflow logs local run metadata by default, while RAGAS and DSPy are opt-in because they can trigger additional LLM calls and cost.

Neuro-symbolic verification techniques such as enum validation, graph constraints, and logic solvers can be important when agent outputs drive pricing, network automation, IT automation, or other high-stakes decisions. That level of formal verification is outside this notebook's scope, but the risk review calls out where stronger validation would be needed.

In [ ]:
from __future__ import annotations

import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from anthropic import Anthropic
from dotenv import load_dotenv
from google import genai
from IPython.display import Markdown, display
from openai import OpenAI

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

load_dotenv(repo_root / ".env")


In [ ]:
def parse_prompt_case_limit(raw_value: str | None, default: str = "all") -> int | None:
    raw = default if raw_value is None else raw_value.strip().casefold()
    if raw in {"", "all", "none", "unlimited"}:
        return None
    limit = int(raw)
    if limit <= 0:
        raise ValueError("AIMBT_LIVE_PROMPT_CASE_LIMIT must be positive or 'all'.")
    return limit


def parse_bool_env(name: str, *, default: bool = False) -> bool:
    raw_value = os.getenv(name)
    if raw_value is None:
        return default
    return raw_value.strip().casefold() in {"1", "true", "yes", "y", "on"}


CANDIDATES = [
    {
        "candidate_model_id": "gpt-5-5",
        "display_name": "GPT-5.5",
        "provider": "openai",
        "api_key_env": "OPENAI_API_KEY",
        "model_name": os.getenv("AIMBT_OPENAI_MODEL_NAME", "gpt-5.5"),
    },
    {
        "candidate_model_id": "claude-opus-4-7",
        "display_name": "Claude Opus 4.7",
        "provider": "anthropic",
        "api_key_env": "ANTHROPIC_API_KEY",
        "model_name": os.getenv("AIMBT_ANTHROPIC_MODEL_NAME", "claude-opus-4-7"),
    },
    {
        "candidate_model_id": "gemini-3-1",
        "display_name": "Gemini 3.1",
        "provider": "google",
        "api_key_env": "GOOGLE_API_KEY",
        "model_name": os.getenv("AIMBT_GOOGLE_MODEL_NAME", "gemini-3.1"),
    },
]

PROMPT_CASE_LIMIT = parse_prompt_case_limit(os.getenv("AIMBT_LIVE_PROMPT_CASE_LIMIT"))
PROVIDER_TIMEOUT_SECONDS = float(os.getenv("AIMBT_PROVIDER_TIMEOUT_SECONDS", "120"))
LATENCY_TARGET_MS = float(os.getenv("AIMBT_LATENCY_TARGET_MS", "8000"))
ENABLE_MLFLOW = parse_bool_env("AIMBT_ENABLE_MLFLOW", default=True)
RUN_RAGAS_EVALS = parse_bool_env("AIMBT_RUN_RAGAS_EVALS")
RUN_DSPY_EVALS = parse_bool_env("AIMBT_RUN_DSPY_EVALS")
RAGAS_EVAL_LIMIT = parse_prompt_case_limit(os.getenv("AIMBT_RAGAS_EVAL_LIMIT"), default="3")
DSPY_EVAL_LIMIT = parse_prompt_case_limit(os.getenv("AIMBT_DSPY_EVAL_LIMIT"), default="3")
MLFLOW_EXPERIMENT_NAME = os.getenv("AIMBT_MLFLOW_EXPERIMENT_NAME", "ai-live-benchmark-notebook")
MLFLOW_TRACKING_URI = os.getenv("AIMBT_MLFLOW_TRACKING_URI") or (repo_root / "mlruns").as_uri()
DSPY_MODEL_NAME = os.getenv("AIMBT_DSPY_MODEL_NAME") or f"openai/{CANDIDATES[0]['model_name']}"
RUN_STARTED_AT = datetime.now(timezone.utc).isoformat()

if "GOOGLE_API_KEY" in os.environ and "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]

missing_credentials = [item["api_key_env"] for item in CANDIDATES if not os.getenv(item["api_key_env"])]
if missing_credentials:
    raise RuntimeError(
        "Missing required live API credential environment variable(s): "
        + ", ".join(missing_credentials)
        + ". Add them to .env or your shell before running this notebook."
    )

display(Markdown("## Run Configuration"))
display(
    pd.DataFrame(
        [
            {
                "candidate_model_id": item["candidate_model_id"],
                "display_name": item["display_name"],
                "provider": item["provider"],
                "provider_api_model_name": item["model_name"],
            }
            for item in CANDIDATES
        ]
    )
)
display(
    pd.DataFrame(
        [
            {"setting": "run_mode", "value": "live_provider"},
            {"setting": "prompt_case_limit", "value": PROMPT_CASE_LIMIT if PROMPT_CASE_LIMIT is not None else "all"},
            {"setting": "provider_timeout_seconds", "value": PROVIDER_TIMEOUT_SECONDS},
            {"setting": "latency_target_ms", "value": LATENCY_TARGET_MS},
            {"setting": "enable_mlflow", "value": ENABLE_MLFLOW},
            {"setting": "mlflow_tracking_uri", "value": MLFLOW_TRACKING_URI},
            {"setting": "mlflow_experiment_name", "value": MLFLOW_EXPERIMENT_NAME},
            {"setting": "run_ragas_evals", "value": RUN_RAGAS_EVALS},
            {"setting": "ragas_eval_limit", "value": RAGAS_EVAL_LIMIT if RAGAS_EVAL_LIMIT is not None else "all"},
            {"setting": "run_dspy_evals", "value": RUN_DSPY_EVALS},
            {"setting": "dspy_eval_limit", "value": DSPY_EVAL_LIMIT if DSPY_EVAL_LIMIT is not None else "all"},
            {"setting": "dspy_model_name", "value": DSPY_MODEL_NAME},
            {"setting": "run_started_at", "value": RUN_STARTED_AT},
        ]
    )
)


In [ ]:
PROMPT_SUITE = [
    {
        "prompt_case_id": "swe-verified-issue-repair",
        "category": "coding",
        "benchmark_refs": "SWE-bench Verified",
        "prompt": "A Python package fails when a config file contains a blank line because parse_config assumes every line contains '='. Describe the bug, propose a minimal patch, and include one regression test case.",
        "expected_terms": ["blank", "line", "skip", "test"],
    },
    {
        "prompt_case_id": "swe-pro-pagination-regression",
        "category": "coding",
        "benchmark_refs": "SWE-bench Pro",
        "prompt": "A web service intermittently returns duplicate records after a pagination refactor. Give a debugging plan, the likely boundary-condition bug, and a concise test that would catch it.",
        "expected_terms": ["pagination", "boundary", "duplicate", "test"],
    },
    {
        "prompt_case_id": "code-generation-balanced-prefix",
        "category": "coding",
        "benchmark_refs": "HumanEval / LiveCodeBench",
        "prompt": "Write a Python function longest_balanced_prefix(text) that returns the longest prefix where parentheses are balanced. Include short examples for empty input, balanced input, and an early unmatched closing parenthesis.",
        "expected_terms": ["def", "longest_balanced_prefix", "balance", "prefix"],
    },
    {
        "prompt_case_id": "math-abstract-rule",
        "category": "reasoning",
        "benchmark_refs": "AIME / ARC-AGI-2",
        "prompt": "A machine transforms each row of three numbers by replacing the third number with the sum of the first two minus their greatest common divisor. For rows (6, 10, ?), (8, 12, ?), and (9, 15, ?), compute the missing values and explain the rule.",
        "expected_terms": ["14", "16", "21", "gcd"],
    },
    {
        "prompt_case_id": "science-qa-constant-volume",
        "category": "reasoning",
        "benchmark_refs": "GPQA Diamond",
        "prompt": "A sealed ideal-gas container is heated while volume stays constant. Which quantity must increase: pressure, volume, mole count, or gas constant? Answer with the quantity and a one-sentence justification.",
        "expected_terms": ["pressure", "constant", "volume", "temperature"],
    },
]

selected_prompt_suite = PROMPT_SUITE if PROMPT_CASE_LIMIT is None else PROMPT_SUITE[:PROMPT_CASE_LIMIT]

display(Markdown("## Prompt Suite"))
display(
    pd.DataFrame(selected_prompt_suite)[
        ["prompt_case_id", "category", "benchmark_refs", "prompt", "expected_terms"]
    ]
)


In [ ]:
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], timeout=PROVIDER_TIMEOUT_SECONDS)
anthropic_client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
google_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])


def _safe_usage_dict(usage_obj: Any) -> dict[str, Any]:
    if usage_obj is None:
        return {}
    if hasattr(usage_obj, "model_dump"):
        return usage_obj.model_dump()
    if hasattr(usage_obj, "to_json_dict"):
        return usage_obj.to_json_dict()
    values: dict[str, Any] = {}
    for name in ("prompt_tokens", "completion_tokens", "total_tokens", "input_tokens", "output_tokens"):
        if hasattr(usage_obj, name):
            values[name] = getattr(usage_obj, name)
    return values


def call_openai(model_name: str, prompt: str) -> tuple[str, dict[str, Any]]:
    response = openai_client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        timeout=PROVIDER_TIMEOUT_SECONDS,
    )
    text = response.choices[0].message.content or ""
    return text, {"usage": _safe_usage_dict(getattr(response, "usage", None))}


def call_anthropic(model_name: str, prompt: str) -> tuple[str, dict[str, Any]]:
    response = anthropic_client.messages.create(
        model=model_name,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
        timeout=PROVIDER_TIMEOUT_SECONDS,
    )
    text_parts = [block.text for block in response.content if getattr(block, "type", None) == "text"]
    return "\n".join(text_parts), {"usage": _safe_usage_dict(getattr(response, "usage", None))}


def call_google(model_name: str, prompt: str) -> tuple[str, dict[str, Any]]:
    response = google_client.models.generate_content(
        model=model_name,
        contents=prompt,
        config={"temperature": 0},
    )
    text = getattr(response, "text", None) or ""
    return text, {"usage": _safe_usage_dict(getattr(response, "usage_metadata", None))}


PROVIDER_CALLS = {
    "openai": call_openai,
    "anthropic": call_anthropic,
    "google": call_google,
}


In [ ]:
invocation_rows: list[dict[str, Any]] = []

for candidate in CANDIDATES:
    provider_call = PROVIDER_CALLS[candidate["provider"]]
    for prompt_case in selected_prompt_suite:
        started = time.perf_counter()
        status = "succeeded"
        response_text = ""
        error_type = None
        error_message = None
        metadata: dict[str, Any] = {}
        try:
            response_text, metadata = provider_call(candidate["model_name"], prompt_case["prompt"])
        except Exception as exc:  # display provider errors as data, not notebook crashes
            status = "failed"
            error_type = type(exc).__name__
            error_message = str(exc)
        latency_ms = (time.perf_counter() - started) * 1000
        invocation_rows.append(
            {
                "candidate_model_id": candidate["candidate_model_id"],
                "display_name": candidate["display_name"],
                "provider": candidate["provider"],
                "provider_api_model_name": candidate["model_name"],
                "prompt_case_id": prompt_case["prompt_case_id"],
                "benchmark_refs": prompt_case["benchmark_refs"],
                "category": prompt_case["category"],
                "status": status,
                "latency_ms": round(latency_ms, 1),
                "response_text": response_text,
                "response_preview": response_text[:500],
                "error_type": error_type,
                "error_message": error_message,
                "usage": metadata.get("usage", {}),
            }
        )

invocations_df = pd.DataFrame(invocation_rows)
display(Markdown("## Live Invocation Results"))
display(
    invocations_df[
        [
            "candidate_model_id",
            "provider_api_model_name",
            "prompt_case_id",
            "benchmark_refs",
            "status",
            "latency_ms",
            "error_type",
            "error_message",
            "response_preview",
        ]
    ]
)


In [ ]:
expected_terms_by_case = {case["prompt_case_id"]: case["expected_terms"] for case in selected_prompt_suite}


def score_expected_terms(response_text: str, expected_terms: list[str]) -> tuple[float, list[str]]:
    if not expected_terms:
        return 0.0, []
    normalized = response_text.casefold()
    found = [term for term in expected_terms if term.casefold() in normalized]
    return len(found) / len(expected_terms), found


score_rows: list[dict[str, Any]] = []
for row in invocation_rows:
    expected_terms = expected_terms_by_case[row["prompt_case_id"]]
    quality_score, matched_terms = score_expected_terms(row["response_text"], expected_terms)
    if row["status"] != "succeeded":
        quality_score = 0.0
        matched_terms = []
    score_rows.append(
        {
            **{key: row[key] for key in ["candidate_model_id", "prompt_case_id", "benchmark_refs", "category", "status", "latency_ms"]},
            "quality_score": round(quality_score, 3),
            "matched_terms": ", ".join(matched_terms),
            "expected_terms": ", ".join(expected_terms),
        }
    )

scores_df = pd.DataFrame(score_rows)
display(Markdown("## Automated Output Quality Observations"))
display(scores_df)

quality_summary_df = (
    scores_df.groupby("candidate_model_id", as_index=False)
    .agg(live_prompt_quality=("quality_score", "mean"))
    .assign(live_prompt_quality=lambda data: data["live_prompt_quality"].round(3))
)
display(Markdown("### Quality Summary"))
display(quality_summary_df)


In [ ]:
ragas_scores_df = pd.DataFrame()
display(Markdown("## RAGAS Evaluation (Optional)"))

if not RUN_RAGAS_EVALS:
    display(
        Markdown(
            "RAGAS evaluation is installed but skipped by default because it can make extra evaluator LLM calls. "
            "Set `AIMBT_RUN_RAGAS_EVALS=true` to run it."
        )
    )
else:
    successful_invocations = invocations_df[invocations_df["status"] == "succeeded"].copy()
    if RAGAS_EVAL_LIMIT is not None:
        successful_invocations = successful_invocations.head(RAGAS_EVAL_LIMIT)

    prompt_by_id = {case["prompt_case_id"]: case["prompt"] for case in selected_prompt_suite}
    ragas_metadata_df = successful_invocations[["candidate_model_id", "prompt_case_id"]].reset_index(drop=True)
    ragas_new_rows = []
    ragas_legacy_rows = []
    for row in successful_invocations.to_dict("records"):
        prompt_text = prompt_by_id[row["prompt_case_id"]]
        reference_text = "Expected concepts: " + ", ".join(expected_terms_by_case[row["prompt_case_id"]])
        ragas_new_rows.append(
            {
                "user_input": prompt_text,
                "response": row["response_text"],
                "retrieved_contexts": [prompt_text, reference_text],
                "reference": reference_text,
            }
        )
        ragas_legacy_rows.append(
            {
                "question": prompt_text,
                "answer": row["response_text"],
                "contexts": [prompt_text, reference_text],
                "ground_truth": reference_text,
            }
        )

    try:
        from ragas import evaluate

        try:
            from ragas import EvaluationDataset
            from ragas.metrics import Faithfulness, ResponseRelevancy

            ragas_dataset = EvaluationDataset.from_list(ragas_new_rows)
            ragas_metrics = [ResponseRelevancy(), Faithfulness()]
        except Exception:
            from datasets import Dataset
            from ragas.metrics import answer_relevancy, faithfulness

            ragas_dataset = Dataset.from_list(ragas_legacy_rows)
            ragas_metrics = [answer_relevancy, faithfulness]

        ragas_result = evaluate(ragas_dataset, metrics=ragas_metrics)
        if hasattr(ragas_result, "to_pandas"):
            ragas_scores_df = ragas_result.to_pandas()
        else:
            ragas_scores_df = pd.DataFrame(ragas_result)
        ragas_scores_df = pd.concat(
            [ragas_metadata_df, ragas_scores_df.reset_index(drop=True)], axis=1
        )
    except Exception as exc:
        ragas_scores_df = pd.DataFrame(
            [
                {
                    "status": "failed",
                    "error_type": type(exc).__name__,
                    "error_message": str(exc),
                    "note": "RAGAS setup varies by version and may require evaluator LLM configuration.",
                }
            ]
        )

    display(ragas_scores_df)


In [ ]:
dspy_scores_df = pd.DataFrame()
display(Markdown("## DSPy Prompt Program Evaluation (Optional)"))

if not RUN_DSPY_EVALS:
    display(
        Markdown(
            "DSPy is installed but skipped by default because it makes extra LLM calls through a DSPy program. "
            "Set `AIMBT_RUN_DSPY_EVALS=true` to run it."
        )
    )
else:
    try:
        import dspy

        dspy_cases = selected_prompt_suite if DSPY_EVAL_LIMIT is None else selected_prompt_suite[:DSPY_EVAL_LIMIT]
        dspy_lm = dspy.LM(DSPY_MODEL_NAME)
        dspy.configure(lm=dspy_lm)

        class BenchmarkAnswer(dspy.Signature):
            """Answer a benchmark prompt clearly and concisely."""

            prompt: str = dspy.InputField(desc="Benchmark prompt")
            answer: str = dspy.OutputField(desc="Answer to evaluate")

        predictor = dspy.Predict(BenchmarkAnswer)
        dspy_rows = []
        for prompt_case in dspy_cases:
            started = time.perf_counter()
            prediction = predictor(prompt=prompt_case["prompt"])
            latency_ms = (time.perf_counter() - started) * 1000
            answer_text = getattr(prediction, "answer", None) or str(prediction)
            quality_score, matched_terms = score_expected_terms(answer_text, prompt_case["expected_terms"])
            dspy_rows.append(
                {
                    "dspy_model_name": DSPY_MODEL_NAME,
                    "prompt_case_id": prompt_case["prompt_case_id"],
                    "benchmark_refs": prompt_case["benchmark_refs"],
                    "latency_ms": round(latency_ms, 1),
                    "quality_score": round(quality_score, 3),
                    "matched_terms": ", ".join(matched_terms),
                    "response_preview": answer_text[:500],
                }
            )
        dspy_scores_df = pd.DataFrame(dspy_rows)
    except Exception as exc:
        dspy_scores_df = pd.DataFrame(
            [
                {
                    "status": "failed",
                    "error_type": type(exc).__name__,
                    "error_message": str(exc),
                    "note": "DSPy requires a supported LM identifier such as openai/<model> and matching credentials.",
                }
            ]
        )

    display(dspy_scores_df)


In [ ]:
operational_rows: list[dict[str, Any]] = []
for candidate_id, group in invocations_df.groupby("candidate_model_id"):
    succeeded = int((group["status"] == "succeeded").sum())
    failed = int((group["status"] != "succeeded").sum())
    invocation_count = int(len(group))
    successful_latencies = group.loc[group["status"] == "succeeded", "latency_ms"]
    operational_rows.append(
        {
            "candidate_model_id": candidate_id,
            "invocation_count": invocation_count,
            "succeeded_count": succeeded,
            "failed_count": failed,
            "success_rate": round(succeeded / invocation_count, 3) if invocation_count else 0.0,
            "error_rate": round(failed / invocation_count, 3) if invocation_count else 0.0,
            "latency_ms_avg": round(float(successful_latencies.mean()), 1) if not successful_latencies.empty else None,
            "latency_ms_p50": round(float(successful_latencies.quantile(0.50)), 1) if not successful_latencies.empty else None,
            "latency_ms_p95": round(float(successful_latencies.quantile(0.95)), 1) if not successful_latencies.empty else None,
        }
    )

operational_df = pd.DataFrame(operational_rows)
display(Markdown("## Operational Metrics"))
display(operational_df)


In [ ]:
BENCHMARK_EVIDENCE = [
    {"candidate_model_id": "gemini-3-1", "benchmark_name": "SWE-bench Verified", "score": 53.8, "notes": "Editable public benchmark input."},
    {"candidate_model_id": "gemini-3-1", "benchmark_name": "SWE-bench Pro", "score": 41.2, "notes": "Editable public benchmark input."},
    {"candidate_model_id": "gemini-3-1", "benchmark_name": "HumanEval / LiveCodeBench", "score": 89.7, "notes": "Supplemental coding signal."},
    {"candidate_model_id": "gemini-3-1", "benchmark_name": "AIME / ARC-AGI-2", "score": 72.4, "notes": "Reasoning signal."},
    {"candidate_model_id": "gemini-3-1", "benchmark_name": "GPQA Diamond", "score": 76.1, "notes": "Science-heavy domain signal."},
    {"candidate_model_id": "claude-opus-4-7", "benchmark_name": "SWE-bench Verified", "score": 62.4, "notes": "Editable public benchmark input."},
    {"candidate_model_id": "claude-opus-4-7", "benchmark_name": "SWE-bench Pro", "score": 48.6, "notes": "Editable public benchmark input."},
    {"candidate_model_id": "claude-opus-4-7", "benchmark_name": "HumanEval / LiveCodeBench", "score": 91.8, "notes": "Supplemental coding signal."},
    {"candidate_model_id": "claude-opus-4-7", "benchmark_name": "AIME / ARC-AGI-2", "score": 74.9, "notes": "Reasoning signal."},
    {"candidate_model_id": "claude-opus-4-7", "benchmark_name": "GPQA Diamond", "score": 79.3, "notes": "Science-heavy domain signal."},
    {"candidate_model_id": "gpt-5-5", "benchmark_name": "SWE-bench Verified", "score": 65.7, "notes": "Editable public benchmark input."},
    {"candidate_model_id": "gpt-5-5", "benchmark_name": "SWE-bench Pro", "score": 51.4, "notes": "Editable public benchmark input."},
    {"candidate_model_id": "gpt-5-5", "benchmark_name": "HumanEval / LiveCodeBench", "score": 93.1, "notes": "Supplemental coding signal."},
    {"candidate_model_id": "gpt-5-5", "benchmark_name": "AIME / ARC-AGI-2", "score": 78.6, "notes": "Reasoning signal."},
    {"candidate_model_id": "gpt-5-5", "benchmark_name": "GPQA Diamond", "score": 81.2, "notes": "Science-heavy domain signal."},
]

BENCHMARK_SOURCE_NOTES = {
    "SWE-bench Verified": "Replace with the current SWE-bench Verified leaderboard or provider benchmark report used for your submission.",
    "SWE-bench Pro": "Replace with the current SWE-bench Pro leaderboard or provider benchmark report used for your submission.",
    "HumanEval / LiveCodeBench": "Replace with the current HumanEval and/or LiveCodeBench source used for your submission.",
    "AIME / ARC-AGI-2": "Replace with the current AIME and/or ARC-AGI-2 source used for your submission.",
    "GPQA Diamond": "Replace with the current GPQA Diamond source used for your submission.",
}

benchmark_df = pd.DataFrame(BENCHMARK_EVIDENCE)
benchmark_df["source_note"] = benchmark_df["benchmark_name"].map(BENCHMARK_SOURCE_NOTES)
benchmark_df["input_type"] = "editable static benchmark evidence"
benchmark_summary_df = (
    benchmark_df.groupby("candidate_model_id", as_index=False)
    .agg(public_benchmark_score=("score", "mean"))
    .assign(public_benchmark_score=lambda data: (data["public_benchmark_score"] / 100).round(3))
)

display(Markdown("## Public Benchmark Evidence Inputs"))
display(Markdown("These are editable static inputs in the notebook. Replace the scores and source notes with current cited values if your assignment requires sourced benchmark data."))
display(benchmark_df)
display(Markdown("### Benchmark Evidence Summary"))
display(benchmark_summary_df)


In [ ]:
WEIGHTS = {
    "live_prompt_quality": 0.40,
    "public_benchmark_score": 0.25,
    "reliability": 0.20,
    "latency_fit": 0.10,
    "risk_adjustment": 0.05,
}


def latency_fit_score(latency_ms_avg: float | None) -> float:
    if latency_ms_avg is None or pd.isna(latency_ms_avg) or latency_ms_avg <= 0:
        return 0.0
    return min(1.0, LATENCY_TARGET_MS / latency_ms_avg)


decision_input_df = (
    operational_df.merge(quality_summary_df, on="candidate_model_id", how="left")
    .merge(benchmark_summary_df, on="candidate_model_id", how="left")
    .fillna({"live_prompt_quality": 0.0, "public_benchmark_score": 0.0})
)
decision_input_df["reliability"] = decision_input_df["success_rate"]
decision_input_df["latency_fit"] = decision_input_df["latency_ms_avg"].apply(latency_fit_score).round(3)
decision_input_df["risk_adjustment"] = (1.0 - decision_input_df["error_rate"]).clip(lower=0.0, upper=1.0).round(3)

for criterion, weight in WEIGHTS.items():
    decision_input_df[f"{criterion}_weighted"] = (decision_input_df[criterion] * weight).round(4)

weighted_columns = [f"{criterion}_weighted" for criterion in WEIGHTS]
decision_input_df["weighted_total"] = decision_input_df[weighted_columns].sum(axis=1).round(4)
decision_matrix_df = decision_input_df.sort_values("weighted_total", ascending=False).reset_index(drop=True)
decision_matrix_df.insert(0, "rank", range(1, len(decision_matrix_df) + 1))

display(Markdown("## Decision Matrix"))
display(pd.DataFrame([{"criterion": key, "weight": value} for key, value in WEIGHTS.items()]))
display(
    decision_matrix_df[
        [
            "rank",
            "candidate_model_id",
            "weighted_total",
            "live_prompt_quality",
            "public_benchmark_score",
            "reliability",
            "latency_fit",
            "risk_adjustment",
            "success_rate",
            "error_rate",
            "latency_ms_avg",
        ]
    ]
)

recommended_model_id = decision_matrix_df.iloc[0]["candidate_model_id"] if not decision_matrix_df.empty else None
display(Markdown(f"### Recommendation\n\nRecommended model: **{recommended_model_id or 'none'}**"))


In [ ]:
provider_failures = invocations_df[invocations_df["status"] != "succeeded"]
risk_rows = [
    {
        "risk": "Hallucinated or overconfident answers",
        "evidence_trigger": "Free-form responses can satisfy keyword checks while still being incomplete.",
        "mitigation": "Review high-impact outputs manually and expand the rubric with task-specific checks before production use.",
    },
    {
        "risk": "Safety refusal mismatch",
        "evidence_trigger": "Providers can differ in when they refuse or comply.",
        "mitigation": "Add policy-specific prompts and inspect refusal behavior for your deployment domain.",
    },
    {
        "risk": "Privacy, logging, and retention exposure",
        "evidence_trigger": "Live prompts are sent to external APIs.",
        "mitigation": "Do not include secrets or customer data in prompts; review each provider's retention and logging settings.",
    },
    {
        "risk": "Benchmark overfitting or contamination",
        "evidence_trigger": "Public benchmark scores may not predict your private workload.",
        "mitigation": "Use the live prompt results and your own private prompts as the stronger decision signal.",
    },
]
if not provider_failures.empty:
    risk_rows.append(
        {
            "risk": "Provider endpoint or model-ID mismatch",
            "evidence_trigger": f"{len(provider_failures)} live invocation(s) failed.",
            "mitigation": "Check provider dashboards and update AIMBT_*_MODEL_NAME values to account-enabled API model IDs.",
        }
    )

display(Markdown("## Responsible AI Risk Review"))
display(pd.DataFrame(risk_rows))


In [ ]:
mlflow_run_id = None
mlflow_status = "skipped"
display(Markdown("## MLflow Tracking"))

if not ENABLE_MLFLOW:
    display(Markdown("MLflow logging skipped because `AIMBT_ENABLE_MLFLOW=false`."))
else:
    try:
        import mlflow

        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
        mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
        run_name = "live-benchmark-" + RUN_STARTED_AT.replace(":", "-")

        with mlflow.start_run(run_name=run_name) as active_run:
            mlflow_run_id = active_run.info.run_id
            mlflow.log_params(
                {
                    "run_mode": "live_provider",
                    "prompt_case_count": len(selected_prompt_suite),
                    "candidate_count": len(CANDIDATES),
                    "latency_target_ms": LATENCY_TARGET_MS,
                    "run_ragas_evals": RUN_RAGAS_EVALS,
                    "run_dspy_evals": RUN_DSPY_EVALS,
                    "dspy_model_name": DSPY_MODEL_NAME,
                }
            )
            for row in decision_matrix_df.to_dict("records"):
                prefix = "model_" + "".join(
                    char if char.isalnum() else "_" for char in row["candidate_model_id"]
                )
                for metric_name in [
                    "weighted_total",
                    "live_prompt_quality",
                    "public_benchmark_score",
                    "reliability",
                    "latency_fit",
                    "risk_adjustment",
                    "success_rate",
                    "error_rate",
                    "latency_ms_avg",
                ]:
                    metric_value = row.get(metric_name)
                    if metric_value is not None and not pd.isna(metric_value):
                        mlflow.log_metric(f"{prefix}_{metric_name}", float(metric_value))

            mlflow_payload_dir = repo_root / "artifacts" / "mlflow_payloads" / run_name
            mlflow_payload_dir.mkdir(parents=True, exist_ok=True)
            (mlflow_payload_dir / "decision_weights.json").write_text(
                json.dumps(WEIGHTS, indent=2, sort_keys=True), encoding="utf-8"
            )
            (mlflow_payload_dir / "candidate_api_model_names.json").write_text(
                json.dumps(
                    {item["candidate_model_id"]: item["model_name"] for item in CANDIDATES},
                    indent=2,
                    sort_keys=True,
                ),
                encoding="utf-8",
            )

            invocations_for_mlflow = invocations_df.drop(columns=["response_text"]).copy()
            invocations_for_mlflow["usage"] = invocations_for_mlflow["usage"].apply(
                lambda value: json.dumps(value, sort_keys=True)
            )
            artifact_tables = {
                "live_invocations.json": invocations_for_mlflow,
                "quality_scores.json": scores_df,
                "operational_metrics.json": operational_df,
                "benchmark_evidence.json": benchmark_df,
                "decision_matrix.json": decision_matrix_df,
                "risk_review.json": pd.DataFrame(risk_rows),
            }
            if not ragas_scores_df.empty:
                artifact_tables["ragas_scores.json"] = ragas_scores_df
            if not dspy_scores_df.empty:
                artifact_tables["dspy_scores.json"] = dspy_scores_df
            for filename, dataframe in artifact_tables.items():
                dataframe.to_json(
                    mlflow_payload_dir / filename,
                    orient="records",
                    indent=2,
                    default_handler=str,
                )
            mlflow.log_artifacts(str(mlflow_payload_dir), artifact_path="benchmark_payload")

        mlflow_status = "logged"
        display(
            Markdown(
                f"Logged this benchmark run to MLflow experiment `{MLFLOW_EXPERIMENT_NAME}` "
                f"at `{MLFLOW_TRACKING_URI}` with run ID `{mlflow_run_id}`. "
                "The notebook logs metrics plus plain artifact files and does not use MLflow Model Registry APIs."
            )
        )
    except Exception as exc:
        mlflow_status = "failed"
        display(
            pd.DataFrame(
                [
                    {
                        "status": mlflow_status,
                        "error_type": type(exc).__name__,
                        "error_message": str(exc),
                    }
                ]
            )
        )


In [ ]:
source_metadata = [
    {"field": "run_mode", "value": "live_provider"},
    {"field": "run_started_at", "value": RUN_STARTED_AT},
    {"field": "candidate_count", "value": len(CANDIDATES)},
    {"field": "prompt_case_count", "value": len(selected_prompt_suite)},
    {"field": "live_invocation_records", "value": len(invocations_df)},
    {"field": "score_records", "value": len(scores_df)},
    {"field": "benchmark_evidence_records", "value": len(benchmark_df)},
    {"field": "mlflow_enabled", "value": ENABLE_MLFLOW},
    {"field": "mlflow_status", "value": mlflow_status},
    {"field": "mlflow_run_id", "value": mlflow_run_id or "none"},
    {"field": "ragas_enabled", "value": RUN_RAGAS_EVALS},
    {"field": "ragas_records", "value": len(ragas_scores_df)},
    {"field": "dspy_enabled", "value": RUN_DSPY_EVALS},
    {"field": "dspy_records", "value": len(dspy_scores_df)},
    {"field": "recommended_model_id", "value": recommended_model_id},
]

display(Markdown("## Source Metadata"))
display(pd.DataFrame(source_metadata))

display(Markdown("## Raw Responses"))
for row in invocation_rows:
    display(Markdown(f"### {row['candidate_model_id']} / {row['prompt_case_id']}"))
    if row["status"] == "succeeded":
        display(Markdown(row["response_text"] or "_No text returned._"))
    else:
        display(Markdown(f"**Error:** `{row['error_type']}` - {row['error_message']}"))
